In [ ]:
# import settings and functions
%run ./../../imports.ipynb


## What mesh?

Copy your choice to the next cell

for SquareTop:
```
analytical_solution_tag = "-ana_square_top"
generate_config = generateConfig_squareTop
generate_mesh = generateMesh_squareTop
```

for SquareSinCos:
```
analytical_solution_tag = "-ana_square_sincos"
generate_config = generateConfig_squareSinCos
generate_mesh = generateMesh_squareSinCos
```

In [ ]:
# Change according to instruction above
analytical_solution_tag = "-ana_mexi_hat"
generate_config = generateConfig_squareMexiHat
generate_mesh = generateMesh_squareMexiHat

# analytical_solution_tag = "-ana_square_top"
# generate_config = generateConfig_squareTop
# generate_mesh = generateMesh_squareTop

# analytical_solution_tag = "-ana_square_sincos"
# generate_config = generateConfig_squareSinCos
# generate_mesh = generateMesh_squareSinCos

## Analysis setup

In [ ]:
# which executable?

exe = hdiv_data_driven_diffusion_snes
sumanalys = "sumanalys.csv"
ana_name = "ana_square_hdiv_dd_mexi_mixed"
prefix = "c5_"

# exe = data_driven_diffusion_snes
# sumanalys = "sumanalys.csv"
# ana_name = "ana_square_dd_mexi_mixed"
# prefix = "c4_"


params.order = 0 # approximation order for temperature
params.order = 1 # approximation order for temperature
# params.order = 2 # approximation order for temperature
# params.order = 3 # approximation order for temperature

# ana_compare_exe = [hdiv_diffusion, classic_diffusion]
# ana_compare_name = ["ana_square_mexi_mixed", "ana_square_mexi_classic"]
# # ana_compare_name = ["ana_square_mexi_mixed"]
# ana_compare_sum = ["sumanalys.csv", "FEM_errors.csv"]

# Convergence analysis parameters
# order_list = [1, 2, 3] # approximation order p
# elem_size_list = [0.5, 0.2, 0.1] # element size h
params.triangle_mesh = True
params.nproc = 1 # number of processors
jumps = ""
if params.nproc == 1:
    jumps = "-get_jumps"
# jumps = "-get_jumps"

run_test = True
run_analysis = True
run_refinement_analysis = True
run_refinement_mesh_analysis = True
run_refinement_hp_analysis = True

# run_test = False
run_analysis = False
run_refinement_analysis = False
run_refinement_mesh_analysis = False
run_refinement_hp_analysis = False

naming = ["order", "gaussnum", "iterations","volume", "datanum","rmsPoiErr", "errorEstimator",
          "L2norm", "H1seminorm","fluxErr", "orderRefinementCounter", "errorIndicatorGrad", "errorIndicatorDiv", "jumpL2", "jumpHdiv", "eleNum"]
# naming = ["order", "gaussnum", "iterations","volume", "datanum","rmsPoiErr", "errorEstimator",
#           "L2norm", "H1seminorm","fluxErr", "orderRefinementCounter"]

error_name_list = ["L2norm", "H1seminorm", "fluxErr"]
error_label_list = [(r'Global error $L^2$-norm'),
               (r'Global error $H^1$-seminorm'), (r'Global Flux error')]

In [ ]:
params.conductivity = 1.0 # linear conductivity
params.element_size = 0.05 # element size in the regular mesh

# params.triangle_mesh = False # use triangular mesh

# Pre-processing parameters
params.mesh_file = "square_mexi"
params.length_x = 1
params.length_y = 1
params.length_z = 0
params.show_mesh = True


# solution parameters
params.log_file = "log" # log file name 


## Run test

In [ ]:
# start display for showing results
display = Display(backend="xvfb", visible=False, size=(1024, 768))
display.start()

In [ ]:
# Testing mesh generation
if run_test:
    params.show_mesh = True
    generate_config(params)
    generate_mesh(params)

In [ ]:
# Testing running analysis

if run_test:
    !rm out*
    params.part_file = params.mesh_file + "_" + str(params.nproc) + "p.h5m"
    !{mofem_part} -my_file {params.mesh_file + ".h5m"} -nparts {params.nproc} -output_file {params.part_file} -dim 2 -adj_dim 1
    !mpirun -np {params.nproc} {exe} -file_name {params.part_file} -my_order {params.order} {analytical_solution_tag} {jumps} -use_line

    !convert.py out*


In [ ]:
# def p_settings(p, params):
#     if not params.p_resolution:
#         params.p_resolution = p.window_size
#     else:
#         p.window_size = params.p_resolution
#     print(f"Current resolution: {params.p_resolution}")
#     font_size = int(params.p_resolution[0] * 0.02 / params.font_page_part)  # Adjust the scaling factor as needed

#     scalar_bar_title = params.show_field_name if params.show_field_name else params.show_field
#     # Define the arguments for the scalar bar
#     scalar_bar_args = {
#         # 'n_labels': 8,
#         'position_x': 0.2,  
#         'position_y': 0.01,  
#         'title': scalar_bar_title,
#         'title_font_size': int(font_size*0.8),
#         'label_font_size': int(font_size*0.6),
#         # "vertical": True,    
#         # "position_x": 0.85,     # Shift scalar bar to the right to create margin
#         # "position_y": 0.2,      # Adjust y-position to lower it slightly
#         # 'fmt': '%.3g'       
#     }

#     return scalar_bar_args

In [ ]:
if run_test:

    colors = [(0, 0, 1), (1, 1, 1), (1, 0, 0)]  # Blue to White to Red
    n_bins = 100
    cmap_name = 'blue_to_red'
    custom_cmap_temperature = LinearSegmentedColormap.from_list(cmap_name, colors, N=n_bins)

    params.show_file = "out_iteration"
    params.show_field = "T"
    params.warp_field_scalar = "T"
    params.show_edges = False
    params.p_cmap = color_temperature
    # params.p_cmap = "RdBu"
    params.clim = (0, 1)
    params.warp_factor = 0.8
    params.p_resolution = (500*3, 400*3)
    params.font_page_part = 1./2.
    params.show_scalar_bar = True
    params.camera_position =  [
            (-1.3, -1.3, 1.1),
            (0.35, 0.35, -0.2),
            (0.25, 0.25, 1),
        ]
    # params.p_cmap = "jet"
    params.p_save = prefix+"T_ord_"+str(params.order)+".pdf"
    show_results(params)


    params.clim = None
    params.camera_position = "xy"
    params.warp_field_scalar = ""


In [ ]:
run_test = True
if run_test:

    colors = [(0, 0, 1), (1, 1, 1), (1, 0, 0)]  # Blue to White to Red
    n_bins = 100
    cmap_name = 'blue_to_red'
    custom_cmap_temperature = LinearSegmentedColormap.from_list(cmap_name, colors, N=n_bins)

    params.show_file = "out_iteration"
    params.show_field = "Q"
    params.warp_field_vector = "Q"

    params.show_edges = False
    params.p_cmap = color_flux
    params.p_cmap = "cool"
    params.clim = (0, 9)
    params.warp_factor = 0.07
    params.p_resolution = (500*3, 400*3)
    params.font_page_part = 1./2.
    params.show_scalar_bar = True
    params.camera_position =  [
            (-1.3, -1.3, 1.1),
            (0.35, 0.35, -0.2),
            (0.25, 0.25, 1),
        ]
    # params.p_cmap = "jet"
    params.p_save = prefix+"Q_ord_"+str(params.order)+".pdf"
    show_results(params)


    params.clim = None
    params.camera_position = "xy"
    params.warp_field_vector = ""


In [ ]:
if run_test and exe == hdiv_data_driven_diffusion_snes:
    params.field_part = 10
    params.show_file = "out_resu"
    params.show_field = "T"
    params.show_field_name = "Grad T"
    params.show_edges = False
    params.clim = [0, 10]
    # params.p_resolution = (1200, 800)
    params.p_resolution = (500*3, 550*3)
    # params.p_resolution = None
    # params.clim = None
    params.p_cmap = color_gradient
    params.p_save = "c5_grad_T_ord_"+str(params.order)+".pdf"
    show_results(params)
    params.field_part = -1
    params.show_field_name = None

In [ ]:
if run_test and exe == hdiv_data_driven_diffusion_snes:
    # params.crop_top_x_percent_pdf = 10
    params.field_part = -1
    params.show_file = "out_resu"
    params.show_field = "G"
    params.show_edges = False
    params.p_cmap = color_gradient
    params.p_save = "c5_G_ord_"+str(params.order)+".pdf"
    show_results(params)

In [ ]:
if run_test and exe == hdiv_data_driven_diffusion_snes:
    # params.crop_top_x_percent_pdf = 10
    params.field_part = -1
    params.show_file = "out_resu"
    params.show_field = "T"
    params.show_edges = False
    # params.p_cmap = 
    params.p_cmap = color_temperature
    params.p_save = "c5_TT_ord_"+str(params.order)+".pdf"
    show_results(params)

In [ ]:
if run_test and exe == hdiv_data_driven_diffusion_snes:
    # params.crop_top_x_percent_pdf = 10
    params.field_part = -1
    params.show_file = "out_resu"
    params.show_field = "Q"
    params.show_edges = False
    params.p_cmap = "cool"
    params.p_save = "c5_QQ_ord_"+str(params.order)+".pdf"
    show_results(params)

In [ ]:
if run_test and exe == hdiv_data_driven_diffusion_snes:
    params.show_file = "out_error"
    params.show_field = "ERROR_INDICATOR_GRAD"
    params.show_edges = False
    params.clim = None
    params.warp_field_scalar = ""
    params.p_cmap = "jet"
    # params.warp_factor = 0.4  # warp factor
    params.p_save = prefix+"err_ind_grad_ord_"+str(params.order)+".pdf"
    show_results(params)

In [ ]:
if run_test and exe == hdiv_data_driven_diffusion_snes:
    params.show_file = "out_error"
    params.show_field = "ERROR_INDICATOR_DIV"
    params.show_edges = False
    params.warp_field_scalar = ""
    # params.warp_factor = 0.4  # warp factor
    params.p_save = prefix+"err_ind_div_ord_"+str(params.order)+".pdf"
    show_results(params)

In [ ]:
if jumps and run_test and exe == hdiv_data_driven_diffusion_snes:
    params.show_file = "out_error"
    params.show_field = "JUMP_L2"
    params.show_edges = False
    params.p_cmap = "jet"
    params.p_save = prefix+"err_ind_jump_ord_"+str(params.order)+".pdf"
    show_results(params)

In [ ]:
if run_test and exe == hdiv_data_driven_diffusion_snes:
    params.show_file = "out_error"
    params.show_field = "ERROR_ESTIMATOR"
    params.show_edges = False
    params.p_cmap = "jet"
    params.p_save = prefix+"err_est_ord_"+str(params.order)+".pdf"
    show_results(params)

In [ ]:
if run_test:
    params.show_file = "out_error"
    params.show_field = "ERROR_H1_SEMINORM"
    params.show_field_2 = "ERROR_L2_NORM"
    params.show_field_3 = "ERROR_FLUX"
    params.show_edges = False
    params.warp_field_scalar = ""
    params.show_field_name = "TOTAL ERROR"
    params.p_save = prefix+"err_total_ord_"+str(params.order)+".pdf"
    show_results(params)

    # unset all set variables
    params.show_field_2 = None
    params.show_field_3 = None
    params.show_field_name = None
    params.show_field = None
    params.p_save = ""